In [ ]:
# !pip install langchain langgraph langchain-openai langchain-core langchain-community langchain-experimental fpdf pdfplumber
# !pip install fpdf pdfplumber

In [ ]:
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import AIMessage
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from pydantic import BaseModel, Field
from fpdf import FPDF
import random
import pdfplumber
import os
import requests
import warnings

warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pwd

/content


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/aiffel/env_keys/.env")

print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY loaded: True


In [ ]:
# LLM 정의
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o",
                temperature=0)

### PDF

In [ ]:
@tool
def read_pdf(file_path: str):
    """
    PDF 파일에서 텍스트를 추출하는 도구입니다.
    표 형식 또는 일반 텍스트가 포함된 PDF를 읽고 문자열로 반환합니다.

    file_path 예시: './report.pdf'
    """
    try:
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text.strip() if text.strip() else "❌ PDF에서 텍스트를 추출할 수 없습니다."
    except Exception as e:
        return f"❌ PDF 읽기 오류: {str(e)}"

In [ ]:
@ tool
def write_pdf(content: str, filename: str = "output.pdf", summary: bool =True):
    """
    텍스트를 PDF 파일로 저장하는 도구입니다.
    PDF형태의 문서로 만들어야할 때 이 도구를 사용하세요.
    """

    if summary:
        prompt = PromptTemplate.from_template("""
                당신은 보고서를 작성하는 어시스턴트입니다. 당신에겐 문서 모음이 제공되고 이를 잘 분석하여 보고서를 작성하여야 합니다.
                아래의 content는 문서 모음입니다. 문서의 제목, 본문을 잘 판단하고 정리하여 요약합니다.
                항상 구조화된 출력을 제공하세요.
                항상 마지막엔 인사이트도 첨부합니다.

                content : {content}
                """)

        chain = prompt | llm

        content = chain.invoke({"content":content}).content

    else:
        pass

    font_url = "https://github.com/google/fonts/raw/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf"
    font_path = "./fonts/NotoSansKR.ttf"

    try:
        os.mkdir("./fonts/")
        response = requests.get(font_url)
        with open(font_path, "wb") as f:
            f.write(response.content)
    except:
        pass

    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    font_path = "/content/fonts/NotoSansKR.ttf"  # <-- 여기에 실제 폰트 파일이 있어야 함

    try:
        pdf.add_font("NotoSans", "", font_path, uni=True)
        pdf.set_font("NotoSans", size=12)
    except:
        raise ValueError("한글 폰트를 등록할 수 없습니다.")

    for line in content.split("\n"):
        pdf.multi_cell(0, 10, line)
    pdf.output(f"./{filename}")

    return f"{filename} 저장 완료"

In [ ]:
# PDF 쓰기 도구
import os
import requests
from fpdf import FPDF
from datetime import datetime
from typing import List, Optional
from PIL import Image


# 1. 기본 FPDF 클래스 상속해서 헤더/푸터 커스터마이징
class PDF(FPDF):
    def __init__(self, font_path="/content/fonts/NotoSansKR.ttf"):
        super().__init__()
        self.title_text = ""
        self.font_path = font_path

    def header(self):
        """페이지 상단에 표시될 헤더"""
        if self.title_text:
            self.set_font('NotoSans', 'B', 16)
            self.cell(0, 10, self.title_text, border=0, ln=1, align='C')
            self.ln(5)

    def footer(self):
        """페이지 하단에 표시될 푸터"""
        self.set_y(-15)
        self.set_font('NotoSans', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', align='C')


def _ensure_font_file(font_path: str):
    os.makedirs(os.path.dirname(font_path), exist_ok=True)
    if not os.path.exists(font_path):
        font_url = "https://github.com/google/fonts/raw/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf"
        resp = requests.get(font_url)
        resp.raise_for_status()
        with open(font_path, "wb") as f:
            f.write(resp.content)


def _register_font(pdf: PDF, font_path: str):
    # 필요 스타일만 등록
    pdf.add_font("NotoSans", "", font_path, uni=True)
    pdf.add_font("NotoSans", "B", font_path, uni=True)
    pdf.add_font("NotoSans", "I", font_path, uni=True)
    pdf.set_font("NotoSans", size=12)


def _add_summary(pdf: PDF, summary: str):
    if not summary:
        return
    pdf.set_font('NotoSans', 'B', 14)
    pdf.cell(0, 10, "Summary", ln=1)
    pdf.set_font('NotoSans', '', 11)
    pdf.multi_cell(0, 6, summary)
    pdf.ln(5)


def _add_content(pdf: PDF, content: str):
    if not content:
        return
    pdf.set_font('NotoSans', 'B', 14)
    pdf.cell(0, 10, "Content", ln=1)
    pdf.set_font('NotoSans', '', 11)
    pdf.multi_cell(0, 6, content)
    pdf.ln(5)


def _add_bullet_list(pdf: PDF, bullets: List[str]):
    if not bullets:
        return
    pdf.set_font('NotoSans', 'B', 14)
    pdf.cell(0, 10, "List", ln=1)
    pdf.set_font('NotoSans', '', 11)

    # 단순 불릿 리스트 구현
    for item in bullets:
        pdf.cell(5)                  # 왼쪽 여백
        pdf.cell(5, 6, u"\u2022")    # 불릿 문자
        pdf.cell(2)                  # 불릿 뒤 간격
        pdf.multi_cell(0, 6, item)
    pdf.ln(3)


def _add_table(pdf: PDF, table: List[List[str]]):
    if not table:
        return
    pdf.set_font('NotoSans', 'B', 14)
    pdf.cell(0, 10, "Table", ln=1)
    pdf.set_font('NotoSans', '', 10)

    # 매우 단순한 고정 폭 테이블 예시
    col_count = len(table[0])
    page_width = pdf.w - 20  # 좌우 마진 고려
    col_width = page_width / col_count
    row_height = 8

    for row_idx, row in enumerate(table):
        # 행 시작 Y를 미리 기억
        start_y = pdf.get_y()
        max_height = row_height  # 필요시 셀 높이 계산해서 최대값 저장

        for col_idx, cell in enumerate(row):
            x = 10 + col_idx * col_width
            # 모든 셀을 같은 Y에서 시작
            pdf.set_xy(x, start_y)
            pdf.multi_cell(col_width, row_height, cell, border=1)

        # 행 전체 렌더링 끝난 후 Y만 이동
        pdf.set_y(start_y + row_height)


def _add_flowchart(pdf: PDF, flowchart_path: Optional[str]):
    if not flowchart_path:
        return
    pdf.add_page()
    pdf.set_font('NotoSans', 'B', 14)
    pdf.cell(0, 10, "Flowchart", ln=1, align='L')
    pdf.ln(5)

    max_w = pdf.w - 30
    max_h = pdf.h - 50

    img = Image.open(flowchart_path)
    iw, ih = img.size

    scale = min(max_w / iw, max_h / ih)
    w = iw * scale
    h = ih * scale

    x = (pdf.w - w) / 2
    y = pdf.get_y()

    pdf.image(flowchart_path, x=x, y=y, w=w, h=h)


def write_pdf_tool(
    content: str,
    filename: str,
    title: str = "",
    summary: str = "",
    bullets: Optional[List[str]] = None,
    table: Optional[List[List[str]]] = None,
    flowchart_path: Optional[str] = None,
):
    """
    PDF 작성 도구:
    - 제목 / 요약 / 본문
    - 목록 (bullets)
    - 표 (table: 2차원 리스트)
    - 플로우차트 (flowchart_path: 이미지 파일 경로)
    """
    if bullets is None:
        bullets = []
    if table is None:
        table = []

    # pdf = PDF()
    # pdf.title_text = title or filename.replace(".pdf", "")
    # pdf.add_page()

    # 1. 폰트 폴더 및 파일 준비
    font_path = "./fonts/NotoSansKR.ttf"
    _ensure_font_file(font_path)

    # 2. PDF 생성 (커스텀 헤더/푸터 + 폰트 설정)
    pdf = PDF(font_path=font_path)
    pdf.title_text = title or filename.replace(".pdf", "")
    _register_font(pdf, font_path)

    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()  # 이 시점에 header에서 NotoSans 사용 가능

    # 작성 시간
    pdf.set_font('NotoSans', 'I', 10)
    pdf.cell(0, 8, f"Created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", ln=1)
    pdf.ln(3)

    # 섹션들 추가
    _add_summary(pdf, summary)

    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    _add_content(pdf, content)
    _add_bullet_list(pdf, bullets)
    _add_table(pdf, table)
    _add_flowchart(pdf, flowchart_path)

    pdf.output(filename)
    return filename


# 2. PDF 생성 함수
def create_structured_pdf(title, summary, content, filename):
    """
    제목, 요약, 내용을 포함한 구조화된 PDF 생성

    Args:
        title (str): 문서 제목
        summary (str): 문서 요약
        content (str): 본문 내용
        filename (str): 저장할 파일명 (예: "report.pdf")
    """

    # 한글 지원을 위해 폰트 설정 (기본 Arial은 한글 미지원)
    # 실제 사용시에는 한글 폰트 파일 경로 필요
    # pdf.add_font('NanumGothic', '', 'NanumGothic.ttf', uni=True)
    # pdf.set_font('NanumGothic', '', 12)

    # 임시로 Arial 사용 (한글은 깨질 수 있음)
    # pdf.set_font('Arial', '', 12)

    # 1. 폰트 폴더 및 파일 준비
    font_url = "https://github.com/google/fonts/raw/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf"
    font_path = "./fonts/NotoSansKR.ttf"

    try:
        os.mkdir("./fonts/")
        response = requests.get(font_url)
        with open(font_path, "wb") as f:
            f.write(response.content)
    except:
        pass

    # 2. PDF 생성 (커스텀 헤더/푸터 + 폰트 설정)
    # font_path = "/content/fonts/NotoSansKR.ttf"  # <-- 여기에 실제 폰트 파일이 있어야 함
    pdf = PDF(font_path=font_path)
    pdf.title_text = title

    try:
        # 한글 폰트 등록 및 기본 폰트 설정
        pdf.add_font("NotoSans", "", font_path, uni=True)
        pdf.set_font("NotoSans", size=12)
        pdf.add_font("NotoSans", "B", font_path,  uni=True)
        pdf.add_font("NotoSans", "I", font_path,  uni=True)
    except:
        raise ValueError("한글 폰트를 등록할 수 없습니다.")


    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    # 제목 (이미 헤더에 표시되므로 여기서는 생략 가능)
    # pdf.set_font('Arial', 'B', 20)
    # pdf.cell(0, 10, title, ln=1, align='C')
    # pdf.ln(10)

    # 작성일시
    # pdf.set_font('Arial', 'I', 10)
    pdf.set_font("NotoSans", size=10)
    pdf.cell(0, 8, f"Created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", ln=1)
    pdf.ln(5)

    # 요약 섹션
    if summary:
        pdf.set_font('NotoSans', 'B', 14)
        pdf.cell(0, 10, "Summary", ln=1)
        pdf.set_font('NotoSans', '', 11)
        pdf.multi_cell(0, 6, summary)
        pdf.ln(5)

    # 구분선
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    # 본문 내용
    pdf.set_font('NotoSans', 'B', 14)
    pdf.cell(0, 10, "Content", ln=1)
    pdf.set_font('NotoSans', '', 11)
    pdf.multi_cell(0, 6, content)

    # PDF 파일 저장
    pdf.output(filename)
    print(f"✅ PDF saved: {filename}")
    return filename


In [ ]:
@ tool
def create_formated_pdf(title, summary, content, filename):
    """제목/요약/본문 내용을 받아 PDF 파일을 생성하고, 생성된 파일 경로를 반환하는 도구."""
    filename = create_structured_pdf(title, summary, content, filename)
    return filename

In [ ]:
# 툴 정의
# TavilySearchResults : 웹 검색 도구
# PythonAstREPLTool : 파이썬 코드 실행 도구
# write_pdf : pdf 생성 도구
# read_pdf : pdf 읽기 도구
# file_delete : 파일 삭제 도구
# list_directory : 파일 목록 읽기 도구

tools = [create_formated_pdf, PythonAstREPLTool(), write_pdf, read_pdf, *FileManagementToolkit(
                                                                            selected_tools=["file_delete","list_directory"]).get_tools()]
formated_write_tool, code_tool, write_tool, read_tool, delete_tool, listdir_tool= tools

In [ ]:
# PDF 제목/요약/내용 쓰기 도구 예시
if __name__ == "__main__":
    create_formated_pdf.invoke({
        "title": "내 이야기 제목",
        "summary": "이 이야기의 요약입니다.",
        "content": "본문 내용이 여기에 들어갑니다.\n\n여러 줄도 가능합니다.",
        "filename": "/content/drive/MyDrive/output.pdf"
    })

✅ PDF saved: /content/drive/MyDrive/output.pdf


In [ ]:
# test write_pdf_tool()

bullets = [
    "Agent는 LangGraph로 상태 기반으로 구성되었습니다.",
    "도구로 Tavily 검색, Python 실행, 파일 관리, PDF 입출력을 사용합니다.",
    "에러 처리 및 재시도 로직이 포함되어 있습니다."
]

table = [
    ["Step", "Action", "Tool"],
    ["1", "사용자 질문 분석", "LLM"],
    ["2", "웹 검색 수행", "TavilySearchResults"],
    ["3", "코드 실행", "PythonAstREPLTool"],
    ["4", "결과 보고서 PDF 생성", "write_pdf_tool"],
]

write_pdf_tool(
    title="Agent Workflow Report",
    summary="에이전트가 사용자의 질의에 응답하기 위해 수행하는 전체 워크플로우를 요약합니다.",
    content="아래는 에이전트의 주요 단계와 각 단계에서 사용하는 도구 목록입니다.",
    bullets=bullets,
    table=table,
    flowchart_path="/content/drive/MyDrive/agent_flowchart.png",  # 미리 준비한 플로우차트 이미지
    filename="/content/drive/MyDrive/agent_workflow_report.pdf",
)


### Read schema

In [ ]:
import json
import os

# data/workflow_data.json 파일 읽기
def load_workflow_schema(filepath='data/workflow_data.json'):
    """
    workflow_data.json 파일을 로드하고 파싱
    """
    try:
        # 파일 존재 확인
        if not os.path.exists(filepath):
            print(f"❌ 파일을 찾을 수 없습니다: {filepath}")
            return None

        # 파일 읽기 (UTF-8 인코딩)
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)

        print("✅ workflow_data.json 로드 성공!")
        print(f"📋 파일 정보: {os.path.getsize(filepath)} bytes")
        print(f"🏷️  Title: {data.get('title', 'N/A')}")
        print(f"📝 Required: {data.get('required', [])}")

        return data

    except json.JSONDecodeError as e:
        print(f"❌ JSON 파싱 오류 (라인 {e.lineno}): {e.msg}")
        return None
    except Exception as e:
        print(f"❌ 파일 읽기 오류: {e}")
        return None

In [ ]:
# 사용 예시
if __name__ == "__main__":

    filepath = "/content/drive/MyDrive/aiffel/Narrative_Logic/DLThon2/data/workflow_data.json"

    schema = load_workflow_schema(filepath)

    if schema:
        # 스키마 구조 출력
        print("\n📊 스키마 구조:")
        agent_persona = schema.get('properties', {}).get('agent_persona', {})
        print(f"  - agent_persona: {list(agent_persona.get('properties', {}).keys())}")

        sections = schema.get('properties', {}).get('sections', {})
        print(f"  - sections type: {sections.get('type', 'N/A')}")

        # sections 배열의 첫 번째 아이템 구조 확인
        if 'items' in sections:
            first_section = sections['items']['properties']
            print(f"  - section properties: {list(first_section.keys())}")


#### 공통 파일 로더

In [ ]:
import json
from pathlib import Path

def load_json(path: str):
    """임의의 JSON 파일 로더"""
    file_path = Path(path).resolve()
    if not file_path.exists():
        raise FileNotFoundError(f"파일 없음: {file_path}")
    with open(file_path, "r", encoding="utf-8-sig") as f:
        return json.load(f)


In [ ]:
def load_workflow_data():
    filepath = "/content/drive/MyDrive/aiffel/Narrative_Logic/DLThon2/data/workflow_data.json"

    return load_json(filepath)

def load_theory_plot():
    filepath = "/content/drive/MyDrive/aiffel/Narrative_Logic/DLThon2/data/theory_plot.json"

    return load_json(filepath)


In [ ]:
# key 트리만 출력하는 헬퍼
from collections.abc import Mapping, Sequence

def print_schema_tree(data, indent=0, max_list_items=3):
    """dict / list 중첩 구조를 트리 형태로 출력"""
    prefix = "  " * indent

    if isinstance(data, Mapping):  # dict 계열
        for key, value in data.items():
            print(f"{prefix}- {key} ({type(value).__name__})")
            print_schema_tree(value, indent + 1, max_list_items)
    elif isinstance(data, Sequence) and not isinstance(data, (str, bytes)):
        # 리스트인 경우, 앞 몇 개만 샘플로 탐색
        for i, item in enumerate(data[:max_list_items]):
            print(f"{prefix}- [idx {i}] ({type(item).__name__})")
            print_schema_tree(item, indent + 1, max_list_items)
        if len(data) > max_list_items:
            print(f"{prefix}  ... (+{len(data) - max_list_items} more)")


In [ ]:
# 사용 예시

workflow_data_schema = load_workflow_data()
theory_plot_schema = load_theory_plot()

print("=== workflow_data_schema 구조 ===")
print_schema_tree(workflow_data_schema)

print("\n\n=== theory_plot_schema 구조 ===")
print_schema_tree(theory_plot_schema)

#### schema_data 별도 처리

In [ ]:
import json
import re
from pathlib import Path

def load_schema_data(path: str):
    text = Path(path).read_text(encoding="utf-8")
    text = text.replace("\r\n", "\n")

    # 0) 줄 단위로 나눠서 "$schema" 포함된 줄은 버린다
    lines = text.split("\n")
    lines = [ln for ln in lines if '"$schema"' not in ln]
    text = "\n".join(lines)

    # 1) 블록 주석 제거: /* ... */
    text = re.sub(r"/\*.*?\*/", "", text, flags=re.DOTALL)

    # 2) 라인 주석 제거: // ...
    text = re.sub(r"//.*", "", text)

    # 3) $ 로 시작하는 라인 제거
    text = re.sub(r"^\s*\$.*$", "", text, flags=re.MULTILINE)

    # 4) 제어문자 제거 (\n, \t 제외)
    text = re.sub(r"[\x00-\x08\x0b-\x0c\x0e-\x1f]", "", text)

    # 5) 빈 줄 제거
    lines = [ln for ln in text.split("\n") if ln.strip() != ""]
    text = "\n".join(lines)

    print("===== cleaned text (first 5 lines) =====")
    for i, ln in enumerate(text.split("\n")[:5], start=1):
        print(f"{i:02d}: {ln}")
    print("========================================")

    return json.loads(text)


In [ ]:
filepath = "/content/drive/MyDrive/aiffel/Narrative_Logic/DLThon2/data/schema_data.json"
schema_data_schema = load_schema_data(filepath)

print("\n\n=== schema_data_schema 구조 ===")
print_schema_tree(schema_data_schema)


### Tool ??